In [2]:
from dotenv import load_dotenv
from pathlib import Path
import os

env_path = Path.cwd().parent.parent / ".env"
load_dotenv(env_path)

api_key = os.getenv("OPENAI_API_KEY")


In [4]:
from dataclasses import dataclass
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command
from rich import print
import uuid

In [5]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage
from typing import Annotated
from operator import add

In [6]:
model = ChatOpenAI(
    model ='gpt-4.1-mini',
    api_key = api_key,
    temperature=0
)

In [ ]:
class State(BaseModel):
    messages:Annotated[list[BaseMessage],add] = Field(..., description="stores all the conversation")
    original_problem_statement:str = Field(..., description="contains the original mathematical expression/problem statement")
    intermediatery_result:str = Field(..., description="contains a single number as the evaluated result of the expression going under operation")
    result_in_transition:float = Field(..., description="contains the intermediate result")
    final_result:float = Field(..., description="Contains the final output")

In [8]:
@tool
def add(*numbers):
    """THis tool sum up the value of all the arguments passed into it."""
    total = 0
    # Loop through the tuple of arguments
    for num in numbers:
        total += num
    return total

In [9]:
@tool
def substract(num1:float, num2:float)->float:
    """ This tool is used to find the difference between the two numbers"""
    return num1-num2

In [10]:
@tool
def multiply(*numbers)->float:
    """ This tool is used to multiply the values of all the arguments passed into it"""
    product=1
    # Loop through multiple arguments
    for num in numbers:
        product = product*num
    return product

In [11]:
@tool
def divide(num1:float, num2:float)->float:
    """THis tool is used to divide the argument passed into it."""
    try:
        if(num2!=0):
            return num1/num2
    except ZeroDivisionError:
        print("Can not divide by 0")

In [ ]:
tools = [add, substract, multiply, divide]
tool_by_name = {tool.name: tool for tool in tools}

system_prompt = """
    You are a helpful mathematical assistant who can solve complex bodmas problems. Use the rules of
    BODMAS (Brackets, Orders, Division and Multiplication, Addition and Subtraction) to solve the problems step by step.

    You have the access of the tool to perform the basic mathematical operations like addition, substraction, multiplication and division.

    In order to solve the problem, you need to break down the problem into smaller parts and use the tools to perform the basic operations.
    You need to keep track of the intermediate results and then use them in the next steps.

    Consider the following rules to solve the problem:
    1. Always start with the innermost brackets and work your way outwards.
    2. The expression going under the operation should be considered as a whole and not broken down into smaller parts.
    3. The result of the evaluated expression should be used in the next steps. 
    2. Perform operations in the order of their precedence (Orders, Division and Multiplication, Addition and Subtraction).
"""

In [16]:
model_with_tools = model.bind_tools(
    tools=tools,
    parallel_tool_calls=False,
    system_message=system_prompt
)